# Weekly Analytics Brief — OLJ — Data Pull

Builds the numbers for the weekly brief. Sections, in build order:
1. **Acquisitions & Churn** (from the daily sheet) — OLJ vs OT, net subscription change
2. GA4 (users, sessions, page views, top articles, countries, sources) — every GA4 metric is split
   **Web / App / Total** (GA4 `platform` dimension: `web` → Web, `iOS` + `Android` → App)
3. CMS (top articles detail, if GA4 doesn't cover it) — *not built yet*

Run top to bottom. The last cell prints the numbers in the same shape as the weekly brief template,
so you can eyeball it before we wire it into the doc.

## Dependencies

Run this once per Colab session (installs are wiped when the runtime resets).

In [ ]:
!pip install -q google-api-python-client google-auth google-auth-oauthlib google-analytics-data python-docx requests

## ⚙️ Settings — paste your CMS API key here

This is the only cell you should need to edit. **On GitHub** the key is read from the repo secret
`CMS_API_KEY` automatically, so nothing needs pasting there. **Easiest in Colab:** click the 🔑 **Secrets** icon in the
left sidebar, add `CMS_API_KEY`, and switch on notebook access. Then the key survives every new copy
of this notebook and you never paste it again. ⚠️ If this notebook ever goes to GitHub, leave the key
**empty** there and add a repo secret called `CMS_API_KEY` instead. The notebook picks it up automatically,
and a key committed to a repo is a leaked key.

In [ ]:
import os

# ---- CMS (new accounts) --------------------------------------------------------------------
CMS_API_KEY = ""   # <-- paste your CMS API key between the quotes

def _colab_secret(name):
    """Colab's 🔑 Secrets panel (left sidebar) -- survives new copies of the notebook, unlike a pasted key."""
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

# Where the key comes from, first match wins:
#   1. GitHub secret CMS_API_KEY  (the workflow passes it in as an env var; nothing to do in the notebook)
#   2. Colab 🔑 Secret CMS_API_KEY
#   3. the line pasted above
_pasted_key = CMS_API_KEY.strip()
for _src, _val in (("GitHub secret / env var", os.environ.get("CMS_API_KEY", "")),
                   ("Colab secret", _colab_secret("CMS_API_KEY")),
                   ("pasted in this cell", _pasted_key)):
    if _val.strip():
        CMS_API_KEY, CMS_KEY_SOURCE = _val.strip(), _src
        break
else:
    CMS_API_KEY, CMS_KEY_SOURCE = "", "none"

if os.environ.get("GITHUB_ACTIONS") == "true" and _pasted_key:
    print("⚠️  A CMS key is pasted in this notebook and it's in the GitHub repo -- delete it from the cell and "
          "rotate the key in the CMS backend; the GitHub secret is all the workflow needs.")

# WhiteBeard's URL format is https://api.{your_domain}/cms/{path} -> for OLJ: https://api.lorientlejour.com/cms/customer
CMS_API_HOST = "api.lorientlejour.com"
CMS_API_HOST = (CMS_API_HOST or os.environ.get("CMS_API_HOST", "")).strip().removeprefix("https://").removeprefix("http://").strip("/")
CMS_BASE_URL = f"https://{CMS_API_HOST}" if CMS_API_HOST else ""

# The backend's /revenue/customer page URL. These 5 columns are all the report needs (keeps pages small);
# "subscriptions" is what section 2b uses to find new subscribers.
# To make the CMS filter by creation date (much faster): apply the creation-date filter on that page,
# copy the URL, paste it here and replace the two dates with {start} and {end}. The notebook fills them
# in with last Monday -> this Sunday.
CMS_BACKEND_URL = (
    "https://managecmsnew.lorientlejour.com/revenue/customer?"
    "columns%5B%5D=id&columns%5B%5D=creationDate&columns%5B%5D=preferredLanguage&columns%5B%5D=source"
    "&columns%5B%5D=subscriptions"
)
CMS_DATE_FMT = "%Y-%m-%d"   # how {start}/{end} are written -- match whatever the backend URL uses

# ---- Email recipients ---------------------------------------------------------------------
# Test runs (you in Colab, or a manual "Run workflow" on GitHub) only go to TEST_RECIPIENTS.
TEST_RECIPIENTS = ["dianafarhat@lorientlejour.com"]
# The scheduled Monday 09:15 send goes to this list. You can also manage it without touching the notebook:
# GitHub repo -> Settings -> Secrets and variables -> Actions -> Variables -> BRIEF_RECIPIENTS
# (comma-separated); when that variable is set, it replaces the list below.
SCHEDULED_RECIPIENTS = [
    "dianafarhat@lorientlejour.com",
    # "someone@lorientlejour.com",
]
EMAIL_AS_BCC = False   # True = recipients don't see each other (and can't reply-all)

print("CMS key:", f"set (from {CMS_KEY_SOURCE})" if CMS_API_KEY else "MISSING", "| API:", (CMS_BASE_URL + "/cms/customer") if CMS_BASE_URL else "host MISSING")

In [2]:
import pandas as pd
import datetime as dt

# --- Week window ---
# Auto-detects the most recently COMPLETED Monday-Sunday week based on today's date.
# To check a specific past week instead, uncomment the override line below and edit it.
today = dt.datetime.combine(dt.date.today(), dt.time.min)
WEEK_END = today - dt.timedelta(days=today.weekday() + 1)  # most recent Sunday before today
# WEEK_END = dt.datetime(2026, 9, 20)  # <- uncomment + edit to override with a specific week

WEEK_START = WEEK_END - dt.timedelta(days=6)  # the Monday of that week
PREV_WEEK_END = WEEK_START - dt.timedelta(days=1)
PREV_WEEK_START = PREV_WEEK_END - dt.timedelta(days=6)

assert WEEK_END.weekday() == 6, "WEEK_END must be a Sunday"

print(f"This week:  {WEEK_START:%Y-%m-%d} (Mon) → {WEEK_END:%Y-%m-%d} (Sun)")
print(f"Last week:  {PREV_WEEK_START:%Y-%m-%d} (Mon) → {PREV_WEEK_END:%Y-%m-%d} (Sun)")


This week:  2026-09-14 (Mon) → 2026-09-20 (Sun)
Last week:  2026-09-07 (Mon) → 2026-09-13 (Sun)


## 0. Read the source sheet (read-only)

Reads values straight from the live Google Sheet via the Sheets API — **no file is downloaded, and
nothing is ever written back**. Uses the `spreadsheets.readonly` OAuth scope, which has no write
methods at all, so there's no code path here that could edit the sheet even by mistake.

**One-time setup** (not part of the weekly run):
1. In Google Cloud Console, create a **service account** and download its JSON key.
2. Share the Google Sheet with that service account's email (looks like
   `something@project-id.iam.gserviceaccount.com`) as **Viewer**.
3. For GitHub Actions: store the JSON key's contents as a repo secret (e.g. `GSHEET_SA_KEY`); the
   workflow writes it to `service_account.json` at run time before this notebook runs. Locally, just
   save the key as `service_account.json` next to this notebook.


In [3]:
import os

SPREADSHEET_ID = "11WU-b3nmvyPlO0fX9VycKgObr-v5-hXTN6ieCv2TOoA"
SERVICE_ACCOUNT_FILE = "service_account.json"

def read_sheet_rows_api(spreadsheet_id, sheet_name, service_account_file, last_col="R", last_row=5000):
    """Reads a tab's rows (from column B, row 3, to last_col/last_row) via the Sheets API.
    Read-only scope — .readonly has no update/append/clear methods, so this can only ever read.
    Returns a list of rows; each row is a list of raw cell values (UNFORMATTED_VALUE, so dates come
    back as serial numbers, same convention Excel uses)."""
    from google.oauth2 import service_account
    from googleapiclient.discovery import build

    creds = service_account.Credentials.from_service_account_file(
        service_account_file,
        scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"],  # read-only, no write methods exist on this scope
    )
    service = build("sheets", "v4", credentials=creds)
    result = service.spreadsheets().values().get(
        spreadsheetId=spreadsheet_id,
        range=f"'{sheet_name}'!B3:{last_col}{last_row}",
        valueRenderOption="UNFORMATTED_VALUE",
    ).execute()
    return result.get("values", [])

def read_sheet_rows_local(path, sheet_name):
    """Fallback for local testing when service_account.json isn't set up yet: reads the same
    B3:R-range shape out of the uploaded snapshot file, so downstream code is identical either way."""
    import openpyxl
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb[sheet_name]
    rows = []
    for r in range(3, ws.max_row + 1):
        row = [ws.cell(row=r, column=c).value for c in range(2, 19)]  # B..R
        if any(v is not None for v in row):
            rows.append(row)
    return rows

USE_API = os.path.exists(SERVICE_ACCOUNT_FILE)
if USE_API:
    print("Reading live sheet via Sheets API (read-only)...")
else:
    print("[no service_account.json found — reading local snapshot instead for now]")


[no service_account.json found — reading local snapshot instead for now]


## 1. Acquisitions & Churn

Rows come from section 0 above (live via API, or local snapshot as fallback) — column indices below
are relative to column B (B=0), matching the B:R range fetched there.

Column layout (confirmed against the sheet's own pre-built totals, so we trust these instead of
re-summing raw columns by hand):
- **Acquisitions**: O = grand total, P = OLJ Basic total, Q = OLJ Premium total, R = OT total → OLJ = P+Q
- **Churns**: L = grand total, M = OLJ Basic total, N = OLJ Premium total, O = OT total → OLJ = M+N


In [4]:
def excel_serial_to_datetime(v):
    """API returns dates as Excel-style serial numbers (UNFORMATTED_VALUE); openpyxl local fallback
    already gives real datetimes. Handle both."""
    if isinstance(v, dt.datetime):
        return v
    if isinstance(v, (int, float)):
        return dt.datetime(1899, 12, 30) + dt.timedelta(days=v)
    return None

def week_totals_from_rows(rows, col_idxs, start, end):
    """col_idxs are 0-based, relative to column B (date is index 0)."""
    totals = {c: 0 for c in col_idxs}
    for row in rows:
        if not row:
            continue
        d = excel_serial_to_datetime(row[0]) if len(row) > 0 else None
        if d and start <= d <= end:
            for c in col_idxs:
                v = row[c] if c < len(row) else 0
                totals[c] += v or 0
    return totals

# 0-based offsets from column B: O=13, P=14, Q=15, R=16 (Acquisitions); L=10, M=11, N=12, O=13 (Churns)
ACQ_COLS = [13, 14, 15, 16]
CHU_COLS = [10, 11, 12, 13]

def acq_churn_week(start, end):
    if USE_API:
        acq_rows = read_sheet_rows_api(SPREADSHEET_ID, "Acquisitions", SERVICE_ACCOUNT_FILE)
        chu_rows = read_sheet_rows_api(SPREADSHEET_ID, "Churns", SERVICE_ACCOUNT_FILE)
    else:
        acq_rows = read_sheet_rows_local("Daily_sheet_Acquisitions_Churns.xlsx", "Acquisitions")
        chu_rows = read_sheet_rows_local("Daily_sheet_Acquisitions_Churns.xlsx", "Churns")

    acq = week_totals_from_rows(acq_rows, ACQ_COLS, start, end)
    chu = week_totals_from_rows(chu_rows, CHU_COLS, start, end)

    olj_new = acq[14] + acq[15]
    ot_new  = acq[16]
    olj_churn = chu[11] + chu[12]
    ot_churn  = chu[13]

    return {
        "olj_new": olj_new, "olj_churn": olj_churn, "olj_net": olj_new - olj_churn,
        "ot_new": ot_new,   "ot_churn": ot_churn,   "ot_net": ot_new - ot_churn,
    }

this_week = acq_churn_week(WEEK_START, WEEK_END)
last_week = acq_churn_week(PREV_WEEK_START, PREV_WEEK_END)

this_week, last_week


({'olj_new': 64,
  'olj_churn': 104,
  'olj_net': -40,
  'ot_new': 25,
  'ot_churn': 13,
  'ot_net': 12},
 {'olj_new': 60,
  'olj_churn': 87,
  'olj_net': -27,
  'ot_new': 24,
  'ot_churn': 20,
  'ot_net': 4})

In [5]:
def net_row(this_w, last_w, prefix):
    return {
        "This week": f"{this_w[f'{prefix}_net']:+d} ({this_w[f'{prefix}_new']} new / {this_w[f'{prefix}_churn']} churn)",
        "Last week": f"{last_w[f'{prefix}_net']:+d} ({last_w[f'{prefix}_new']} new / {last_w[f'{prefix}_churn']} churn)",
        "WoW (Δ net)": f"{this_w[f'{prefix}_net'] - last_w[f'{prefix}_net']:+d}",
    }

olj_table = pd.DataFrame([net_row(this_week, last_week, "olj")], index=["Net Subscription Change (new − churn)"])
ot_table  = pd.DataFrame([net_row(this_week, last_week, "ot")],  index=["Net Subscription Change (new − churn)"])

print("OLJ")
display(olj_table)
print("\nOT")
display(ot_table)


OLJ


,This week,Last week,WoW (Δ net)
Net Subscription Change (new − churn),-40 (64 new / 104 churn),-27 (60 new / 87 churn),-13



OT


,This week,Last week,WoW (Δ net)
Net Subscription Change (new − churn),+12 (25 new / 13 churn),+4 (24 new / 20 churn),+8


## 2. New accounts — CMS (OLJ vs OT, Web / App / Other)

Pulls every account created **last Monday → this Sunday** from `GET /cms/customer` (key and URLs come
from the ⚙️ Settings cell), then classifies each one:

| Field | Rule |
|---|---|
| **Brand** (`preferredLanguage`) | `en` / `english` / `anglais` → **OT** · anything else (blank, `fr`, `french`…) → **OLJ** |
| **Platform** (`source`) | `web_olj`, `web_ot` → **Web** · `app`, `todayapp` → **App** · anything else → **Other** (newsletters etc.) |

- In the scorecard, *New accounts* gets Web and App columns. "Other" is counted in the Total and noted
  under the row label, so Web + App + Other = Total.
- `creationDate` is UTC, so it's converted to **Beirut time** before bucketing into Mon–Sun weeks.
- Counting is always re-checked locally by date, so a missing or wrong server-side filter can't skew it.
  Without a date filter in `CMS_BACKEND_URL` it just pages through more customers (and stops early if
  results come newest-first).
- Each run prints which `source` and `preferredLanguage` values it saw. That makes any value landing in
  "Other" obvious, and you can add it to `SOURCE_PLATFORM` / `OT_LANGUAGES` below.
- If the CMS isn't configured or the call fails, it falls back to the **`Accounts created OLJ OT`**
  sheet tab (totals only, no Web/App split). While both sources work, it prints the two side by side.

In [ ]:
import requests
from collections import Counter
from urllib.parse import urlsplit, parse_qsl
from zoneinfo import ZoneInfo

# ---------- Classification rules (edit here if new values show up) ----------
OT_LANGUAGES = {"en", "eng", "english", "anglais"}          # -> OT; anything else (blank, fr, french...) -> OLJ
SOURCE_PLATFORM = {"web_olj": "Web", "web_ot": "Web",       # -> Web
                   "app": "App", "todayapp": "App"}         # -> App; anything else -> Other
OTHER_BUCKET = "Other"
ACCOUNT_BUCKETS = ["Web", "App", OTHER_BUCKET]

CMS_TZ = ZoneInfo("Asia/Beirut")
CMS_MAX_PAGES = 2000        # safety stop
CMS_READY = bool(CMS_API_KEY and CMS_BASE_URL)

def _pick(c, *names):
    """First non-empty value for any of `names` (case-insensitive), looking at the top level and then
    inside `fields` / `preferences` -- the API and the backend page don't always name/nest columns alike."""
    wanted = {n.lower() for n in names}
    for scope in (c, c.get("fields"), c.get("preferences")):
        if isinstance(scope, dict):
            for k, v in scope.items():
                if k.lower() in wanted and v not in (None, ""):
                    return v
    return None

def _lang(c):
    return str(_pick(c, "preferredLanguage", "preferred_language", "language") or "").strip().lower()

def _source(c):
    return str(_pick(c, "source", "acquisitionSource", "acquisition_source") or "").strip().lower()

def brand_of(c):
    lang = _lang(c)
    return "ot" if lang in OT_LANGUAGES or lang.startswith(("en-", "en_")) else "olj"

def platform_of(c):
    return SOURCE_PLATFORM.get(_source(c), OTHER_BUCKET)

def _created_local(c):
    """creationDate (UTC ISO string) -> naive Beirut-time datetime, or None."""
    raw = _pick(c, "creationDate", "creation_date", "created")
    if not raw:
        return None
    try:
        d = dt.datetime.fromisoformat(str(raw).replace("Z", "+00:00"))
    except ValueError:
        return None
    if d.tzinfo is None:
        d = d.replace(tzinfo=dt.timezone.utc)
    return d.astimezone(CMS_TZ).replace(tzinfo=None)

def _backend_params(start, end):
    query = urlsplit(CMS_BACKEND_URL).query if "?" in CMS_BACKEND_URL else CMS_BACKEND_URL
    fill = lambda v: v.replace("{start}", start.strftime(CMS_DATE_FMT)).replace("{end}", end.strftime(CMS_DATE_FMT))
    params = [(k, fill(v)) for k, v in parse_qsl(query, keep_blank_values=True) if k != "page" and v != ""]
    has_date_filter = any("{start}" in v for _, v in parse_qsl(query))   # parse_qsl decodes %7Bstart%7D too
    return params, has_date_filter

def cms_fetch_customers(start, end):
    """Every customer the query returns, across all pages (deduped by id). Stops early when there's no
    server-side date filter but results come newest-first and have gone past `start`."""
    base_params, has_date_filter = _backend_params(start, end)
    if not has_date_filter:
        print("[CMS] no creation-date filter in CMS_BACKEND_URL -- fetching and filtering here (slower)")

    session = requests.Session()
    session.headers.update({"API-Key": CMS_API_KEY, "Accept": "application/json"})
    url = CMS_BASE_URL.rstrip("/") + "/cms/customer"

    customers, seen, page = [], set(), None
    newest_first, prev_last = True, None
    for _ in range(CMS_MAX_PAGES):
        params = base_params + ([("page", page)] if page is not None else [])
        resp = session.get(url, params=params, timeout=60)
        resp.raise_for_status()
        payload = resp.json()
        batch = []
        for c in payload.get("data") or []:
            key = str(_pick(c, "userId", "id") or id(c))
            if key not in seen:
                seen.add(key); batch.append(c)
        if not batch:                                   # empty or repeated page -> done
            break
        customers += batch
        total = payload.get("total") or 0
        if total and len(seen) >= total:
            break

        dates = [d for d in map(_created_local, batch) if d]
        if dates:
            chain = ([prev_last] if prev_last else []) + dates
            newest_first = newest_first and all(a >= b for a, b in zip(chain, chain[1:]))
            prev_last = dates[-1]
            if not has_date_filter and newest_first and dates[-1] < start:
                break                                   # sorted newest-first and already past the window

        page = (payload["page"] if payload.get("page") is not None else (page or 0)) + 1
    else:
        print(f"[CMS] stopped after {CMS_MAX_PAGES} pages -- add the creation-date filter to CMS_BACKEND_URL")
    return customers

def cms_accounts_week(customers, start, end):
    """{'olj_new_accounts', 'ot_new_accounts', 'olj_split': {'Web','App','Other'}, 'ot_split': {...}}"""
    counts = {"olj": Counter(), "ot": Counter()}
    for c in customers:
        d = _created_local(c)
        if d is not None and start.date() <= d.date() <= end.date():
            counts[brand_of(c)][platform_of(c)] += 1
    out = {}
    for b in ("olj", "ot"):
        out[f"{b}_split"] = {k: counts[b].get(k, 0) for k in ACCOUNT_BUCKETS}
        out[f"{b}_new_accounts"] = sum(out[f"{b}_split"].values())
    return out

# ---------- Sheet fallback ----------
def read_accounts_created_rows():
    if USE_API:
        return read_sheet_rows_api_custom(SPREADSHEET_ID, "Accounts created OLJ OT",
                                            SERVICE_ACCOUNT_FILE, first_col="A", last_col="D")
    import openpyxl
    wb = openpyxl.load_workbook("Daily_sheet_Acquisitions_Churns.xlsx", data_only=True)
    ws = wb["Accounts created OLJ OT"]
    rows = []
    for r in range(2, ws.max_row + 1):
        row = [ws.cell(row=r, column=c).value for c in range(1, 5)]  # A..D
        if any(v is not None for v in row):
            rows.append(row)
    return rows

def read_sheet_rows_api_custom(spreadsheet_id, sheet_name, service_account_file, first_col="A", last_col="D", last_row=5000):
    """Same as read_sheet_rows_api in section 0, but this tab's date column is A, not B."""
    from google.oauth2 import service_account as sa
    from googleapiclient.discovery import build
    creds = sa.Credentials.from_service_account_file(
        service_account_file, scopes=["https://www.googleapis.com/auth/spreadsheets.readonly"])
    service = build("sheets", "v4", credentials=creds)
    result = service.spreadsheets().values().get(
        spreadsheetId=spreadsheet_id, range=f"'{sheet_name}'!{first_col}2:{last_col}{last_row}",
        valueRenderOption="UNFORMATTED_VALUE").execute()
    return result.get("values", [])

def accounts_created_week_sheet(rows, start, end):
    ot_total, olj_total = 0, 0
    for row in rows:
        if not row:
            continue
        d = excel_serial_to_datetime(row[0]) if len(row) > 0 else None
        if d and start <= d <= end:
            ot_total += (row[1] if len(row) > 1 and row[1] else 0)
            olj_total += (row[2] if len(row) > 2 and row[2] else 0)
    return {"olj_new_accounts": int(olj_total), "ot_new_accounts": int(ot_total), "olj_split": None, "ot_split": None}

# ---------- Build this week / last week ----------
this_week_accounts = last_week_accounts = None
ACCOUNTS_SOURCE = None
NEW_ACCOUNT_IDS = set()   # ids of accounts created last Monday -> this Sunday (used by 2b to spot new subscribers)

if CMS_READY:
    try:
        _customers = cms_fetch_customers(PREV_WEEK_START, WEEK_END)   # one pull covers both weeks
        this_week_accounts = cms_accounts_week(_customers, WEEK_START, WEEK_END)
        last_week_accounts = cms_accounts_week(_customers, PREV_WEEK_START, PREV_WEEK_END)
        ACCOUNTS_SOURCE = "CMS"

        _in_window = [c for c in _customers if (d := _created_local(c)) and PREV_WEEK_START.date() <= d.date() <= WEEK_END.date()]
        NEW_ACCOUNT_IDS = {str(i) for c in _in_window if (i := _pick(c, "userId", "id")) is not None}
        print(f"[CMS] {len(_customers)} fetched, {len(_in_window)} created in the two weeks")
        print("  source values:  ", dict(Counter(_source(c) or "(blank)" for c in _in_window).most_common()))
        print("  language values:", dict(Counter(_lang(c) or "(blank)" for c in _in_window).most_common()))
        if _customers and not _in_window:
            print("  first record's fields (check the names match):", sorted(_customers[0].keys()))
    except Exception as e:
        hint = " (401/403 -> check CMS_API_KEY)" if isinstance(e, requests.HTTPError) else ""
        print(f"[CMS] failed, falling back to the sheet{hint}: {type(e).__name__}: {e}")
else:
    print("[CMS not configured -- fill in CMS_API_KEY / CMS_BASE_URL in ⚙️ Settings; using the sheet for now]")

try:
    _sheet_rows = read_accounts_created_rows()
except Exception as e:
    _sheet_rows = []
    if ACCOUNTS_SOURCE is None:
        print(f"[sheet] couldn't read 'Accounts created OLJ OT': {type(e).__name__}: {e}")

if ACCOUNTS_SOURCE is None:
    this_week_accounts = accounts_created_week_sheet(_sheet_rows, WEEK_START, WEEK_END)
    last_week_accounts = accounts_created_week_sheet(_sheet_rows, PREV_WEEK_START, PREV_WEEK_END)
    ACCOUNTS_SOURCE = "sheet"
elif _sheet_rows:
    _s_this = accounts_created_week_sheet(_sheet_rows, WEEK_START, WEEK_END)
    _s_last = accounts_created_week_sheet(_sheet_rows, PREV_WEEK_START, PREV_WEEK_END)
    print(f"[check] CMS vs sheet -- OLJ this week {this_week_accounts['olj_new_accounts']} vs {_s_this['olj_new_accounts']}, "
          f"last week {last_week_accounts['olj_new_accounts']} vs {_s_last['olj_new_accounts']} | "
          f"OT this week {this_week_accounts['ot_new_accounts']} vs {_s_this['ot_new_accounts']}")

print(f"New accounts source: {ACCOUNTS_SOURCE}")
this_week_accounts, last_week_accounts

### 2b. New subscriptions (acquisitions) — from the new accounts' subscriptions

`/cms/analytics/orders` turned out to be a **dashboard endpoint**: it returns KPI cards and daily chart
series, with no individual orders and no user ids, so the rules below can't be applied to it. Instead,
this reads the `subscriptions` list that `/cms/customer` already returns **for each new account from
section 2**. There's no extra call, and the "is it a new account?" check is built in.

A subscription counts as a **new acquisition** when:

| Check | Rule |
|---|---|
| New account | it belongs to an account created last Monday → this Sunday (section 2) |
| Week | its `activationDate` (Beirut time) falls in that week |
| Paid | its status isn't pending / failed / unpaid (`UNPAID_WORDS` below). Every status value seen is printed, so this is easy to confirm |
| Subscription group | contains `donation` or `integrale` → **not counted** · contains the word `OT` or `English` → **OT** · anything else (OLJ Basic, Premium, French…) → **OLJ** |
| Payment method | Apple (App Store) or Google (Play Store) → **App** · Free (staff) → **not counted** · test payments → **not counted** · everything else (Stripe, MPGS, OMT, Cash…) → **Web** |

- **One acquisition per person per brand per week,** taken from their first qualifying subscription.
- **If the customer rows have no `subscriptions` field,** the cell says so and keeps the sheet numbers from
  section 1.

In [ ]:
import re
import unicodedata

# ---------- Rules (edit here) ----------
EXCLUDED_GROUP_WORDS = ("donation", "integrale")     # accents ignored, so "intégrale" matches too
EXCLUDE_TEST_PAYMENTS = True                         # Stripe Test / MPGS Test
UNPAID_WORDS = ("pending", "fail", "unpaid", "await", "incomplete", "declin", "error", "refus")

GROUP_KEYS  = ("group_name", "groupName", "group")
METHOD_KEYS = ("paymentMethod", "payment_method", "gateway")
STATUS_KEYS = ("status_code_text", "payment_status", "paymentStatus", "status_text", "status")
DATE_KEYS   = ("activationDate", "activation_date", "startDate", "creationDate")

def _norm(s):
    return unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().strip().lower()

def _named(o, keys):
    """First non-empty value among `keys`; if the value is an object ({"id":…, "name":…}), its name."""
    for k in keys:
        for kk, v in o.items():
            if kk.lower() != k.lower() or v in (None, ""):
                continue
            if isinstance(v, dict):
                v = next((v[n] for n in ("name", "title", "label", "description") if v.get(n)), None)
            if v is not None and not isinstance(v, (list, dict)):
                return v
    return None

def sub_brand(group):
    g = _norm(group or "")
    if any(w in g for w in EXCLUDED_GROUP_WORDS):
        return None
    if re.search(r"\bot\b", g) or "english" in g:
        return "ot"
    return "olj"

def sub_platform(method):
    m = _norm(method or "")
    if "free" in m:
        return None                                  # employees
    if EXCLUDE_TEST_PAYMENTS and "test" in m:
        return None
    if any(w in m for w in ("apple", "app store", "appstore", "ios", "google", "play store", "playstore", "android")):
        return "App"
    return "Web"

def sub_is_paid(s):
    st = _norm(_named(s, STATUS_KEYS) if _named(s, STATUS_KEYS) is not None else "")
    return not any(w in st for w in UNPAID_WORDS)

def _local_dt(raw):
    if raw in (None, ""):
        return None
    try:
        if isinstance(raw, (int, float)) or str(raw).isdigit():           # epoch seconds / milliseconds
            ts = float(raw)
            d = dt.datetime.fromtimestamp(ts / 1000 if ts > 1e11 else ts, tz=dt.timezone.utc)
        else:
            d = dt.datetime.fromisoformat(str(raw).strip().replace("Z", "+00:00"))
    except (ValueError, OverflowError, OSError):
        return None
    if d.tzinfo is None:
        d = d.replace(tzinfo=dt.timezone.utc)
    return d.astimezone(CMS_TZ).replace(tzinfo=None)

SAFE_TO_SHOW = {k.lower() for k in GROUP_KEYS + METHOD_KEYS + STATUS_KEYS + DATE_KEYS
                + ("status_code", "deactivationDate", "nextRenewal", "product", "description")}

def _explain_subscriptions(customers):
    """Why nothing matched: what the 'subscriptions' field actually holds. Shows field NAMES, and values only
    for non-personal fields (group, method, status, dates)."""
    vals = [c.get("subscriptions") for c in customers]
    kinds = Counter("empty list" if v == [] else f"list[{len(v)}]" if isinstance(v, list) else
                    "missing/null" if v is None else type(v).__name__ for v in vals)
    print("  'subscriptions' holds:", dict(kinds.most_common(6)))
    sample = next((v for v in vals if v not in (None, [], "", {})), None)
    if sample is None:
        print("  -> every new account's list is empty: /cms/customer probably doesn't include subscriptions in list "
              "results, so we need another endpoint (e.g. a per-customer or subscriptions endpoint in Swagger).")
        return
    first = sample[0] if isinstance(sample, list) and sample else sample
    if isinstance(first, dict):
        print("  first subscription's fields:", sorted(first.keys()))
        print("  its non-personal values:   ", {k: v for k, v in first.items()
                                                if k.lower() in SAFE_TO_SHOW and not isinstance(v, (dict, list))})
        pi = first.get("purchased_item")
        if isinstance(pi, dict):
            print("  purchased_item fields:     ", sorted(pi.keys()))
    else:
        print(f"  it's a {type(first).__name__}, not an object -- e.g. {str(first)[:80]!r}")

def _subscriptions(c):
    subs = c.get("subscriptions")
    return [s for s in subs if isinstance(s, dict)] if isinstance(subs, list) else []

def acquisitions_week(new_customers, start, end):
    """{'olj_new', 'ot_new', 'olj_new_split': {'Web','App'}, 'ot_new_split': {...}}"""
    counts, counted = {"olj": Counter(), "ot": Counter()}, set()
    rows = [(d, str(_pick(c, "userId", "id")), s) for c in new_customers for s in _subscriptions(c)
            if (d := _local_dt(_named(s, DATE_KEYS))) is not None]
    for d, uid, s in sorted(rows, key=lambda r: r[0]):          # earliest first -> first subscription wins
        if not (start.date() <= d.date() <= end.date()) or not sub_is_paid(s):
            continue
        brand, platform = sub_brand(_named(s, GROUP_KEYS)), sub_platform(_named(s, METHOD_KEYS))
        if brand is None or platform is None or (brand, uid) in counted:
            continue
        counted.add((brand, uid))
        counts[brand][platform] += 1
    out = {}
    for b in ("olj", "ot"):
        out[f"{b}_new_split"] = {p: counts[b].get(p, 0) for p in ("Web", "App")}
        out[f"{b}_new"] = sum(out[f"{b}_new_split"].values())
    return out

# ---------- Run ----------
ACQUISITIONS_SOURCE = "sheet"
if ACCOUNTS_SOURCE == "CMS":
    _new_customers = [c for c in _customers if str(_pick(c, "userId", "id")) in NEW_ACCOUNT_IDS]
    if not any("subscriptions" in c for c in _new_customers):
        print("[subscriptions] the customer rows have no 'subscriptions' field -- make sure CMS_BACKEND_URL in "
              "⚙️ Settings includes columns%5B%5D=subscriptions, then re-run 2 and 2b. Using the sheet for now.")
        if _new_customers:
            print("  fields returned:", sorted(_new_customers[0].keys()))
    else:
        _acq_this = acquisitions_week(_new_customers, WEEK_START, WEEK_END)
        _acq_last = acquisitions_week(_new_customers, PREV_WEEK_START, PREV_WEEK_END)

        _win = [s for c in _new_customers for s in _subscriptions(c)
                if (d := _local_dt(_named(s, DATE_KEYS))) and PREV_WEEK_START.date() <= d.date() <= WEEK_END.date()]
        _lbl = lambda v: "not counted" if v is None else v
        print(f"[subscriptions] {len(_new_customers)} new accounts, {len(_win)} subscriptions activated in the two weeks")
        if not _win:
            _explain_subscriptions(_new_customers)
            print("  -> keeping the SHEET numbers (a 0 here means the data wasn't read, not that nobody subscribed). "
                  "Paste these lines back.")
        print("  groups:  ", {f"{g} -> {_lbl(sub_brand(g))}": n
                              for g, n in Counter(str(_named(s, GROUP_KEYS) or "(blank)") for s in _win).most_common()})
        print("  methods: ", {f"{m} -> {_lbl(sub_platform(m))}": n
                              for m, n in Counter(str(_named(s, METHOD_KEYS) or "(blank)") for s in _win).most_common()})
        print("  statuses:", {f"{st} -> {'paid' if sub_is_paid({'status': st}) else 'skipped'}": n
                              for st, n in Counter(str(_named(s, STATUS_KEYS) or "(blank)") for s in _win).most_common()})
        if _win: print(f"[check] CMS vs sheet -- OLJ this week {_acq_this['olj_new']} vs {this_week['olj_new']}, "
              f"last week {_acq_last['olj_new']} vs {last_week['olj_new']} | "
              f"OT this week {_acq_this['ot_new']} vs {this_week['ot_new']}")

        if _win:
            for wk, acq in ((this_week, _acq_this), (last_week, _acq_last)):
                wk.update(acq)
                wk["olj_net"] = wk["olj_new"] - wk["olj_churn"]
                wk["ot_net"] = wk["ot_new"] - wk["ot_churn"]
            ACQUISITIONS_SOURCE = "CMS"
else:
    print("[subscriptions] skipped -- section 2 didn't get the new accounts from the CMS "
          "(check the ⚙️ Settings output says 'CMS key: set', then run 2 and 2b again)")

print(f"New subscriptions source: {ACQUISITIONS_SOURCE}")
{k: this_week[k] for k in ("olj_new", "ot_new")}, {k: last_week[k] for k in ("olj_new", "ot_new")}

## 3. GA4 — Users, Sessions, Page views, Top articles, Countries, Sources

Uses the **GA4 Data API** (`analytics.readonly` scope — same read-only pattern as sections 0 and 2)
with the **same service account** from section 0 — you just need to also grant it access on the GA4
property itself (one extra step below), no new key needed.

**One-time setup:**
1. In the same Google Cloud project as before, enable the **"Google Analytics Data API"**
   (Console → search "Google Analytics Data API" → Enable).
2. In GA4: **Admin → Property Access Management** (for the property, not the account) → **+** →
   add the service account's email (the same `...iam.gserviceaccount.com` address from section 0) →
   role **Viewer**.
3. Find your **Property ID**: GA4 Admin → Property Details → it's a plain number like `123456789`
   (not the "G-XXXXXXX" measurement ID — that's a different thing). Fill it into `GA4_PROPERTY_ID` below.

⚠️ Same caveat as CMS: **untested against a real property** — I don't have GA4 access from here. If
`GA4_PROPERTY_ID` is left as the placeholder, or the service account isn't authorized yet, this section
runs on clearly-labeled **example data** instead, so you can see the shape of the output now and swap
in real numbers once it's connected.

**Fallback while waiting on Property Access Management approval:** logs in as *you* instead of the
service account. First run opens a browser for a one-time "Allow" click; after that it's silent
(the refresh token in `token.json` handles every run after, including in GitHub Actions later — no
repeated clicking).

**One-time setup for this fallback specifically:**
1. Same Google Cloud project → **APIs & Services → Credentials → Create Credentials → OAuth client ID**
   → Application type **Desktop app** → Create.
2. Download its JSON → save as `oauth_client_secret.json` next to this notebook (different file from
   `service_account.json` — don't mix them up).
3. Run the next cell. A browser tab opens asking you to log in and approve — do that once.

⚠️ One thing to know: if the OAuth app stays in **"Testing"** publishing status (Cloud Console →
OAuth consent screen), Google expires the refresh token after 7 days, which would silently break the
scheduled run a week in. Worth switching it to **"In production"** (or **"Internal"** if this is a
Google Workspace org) once you confirm the login flow works, so it keeps running unattended.


In [7]:
GA4_PROPERTY_ID = "328439412"  # GA4 Admin > Property Details
OAUTH_CLIENT_SECRET_FILE = "oauth_client_secret.json"  # from Cloud Console > Credentials > OAuth client ID (Desktop app)
OAUTH_TOKEN_FILE = "token.json"  # created automatically after your first approval; reused silently after that
GA4_SCOPES = ["https://www.googleapis.com/auth/analytics.readonly"]

def get_ga4_credentials():
    """Prefers the service account (once Property Access Management is granted); falls back to
    logging in as you via OAuth, which only needs a browser click on the very first run."""
    if os.path.exists(SERVICE_ACCOUNT_FILE):
        from google.oauth2 import service_account as ga_service_account
        return ga_service_account.Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=GA4_SCOPES)

    if os.path.exists(OAUTH_CLIENT_SECRET_FILE):
        from google.auth.transport.requests import Request
        from google.oauth2.credentials import Credentials
        from google_auth_oauthlib.flow import InstalledAppFlow

        creds = None
        if os.path.exists(OAUTH_TOKEN_FILE):
            creds = Credentials.from_authorized_user_file(OAUTH_TOKEN_FILE, GA4_SCOPES)
        if not creds or not creds.valid:
            if creds and creds.expired and creds.refresh_token:
                creds.refresh(Request())
            else:
                flow = InstalledAppFlow.from_client_secrets_file(OAUTH_CLIENT_SECRET_FILE, GA4_SCOPES)
                creds = flow.run_local_server(port=0)  # opens your browser -- click Allow once
            with open(OAUTH_TOKEN_FILE, "w") as f:
                f.write(creds.to_json())
        return creds

    return None

GA4_READY = (
    (os.path.exists(SERVICE_ACCOUNT_FILE) or os.path.exists(OAUTH_CLIENT_SECRET_FILE))
    and GA4_PROPERTY_ID != "REPLACE_WITH_YOUR_PROPERTY_ID"
)

# Always defined (even as None when GA4 isn't connected yet), so downstream cells can safely pass
# stream_filter=olj_stream_filter / ot_stream_filter regardless of GA4_READY -- avoids a NameError
# in the example-data fallback path, where these never get built.
olj_stream_filter = None  # only OLJ is used in this dedicated script

if GA4_READY:
    from google.analytics.data_v1beta import BetaAnalyticsDataClient
    from google.analytics.data_v1beta.types import (
        RunReportRequest, DateRange, Metric, Dimension, OrderBy,
        FilterExpression, Filter,
    )
    ga4_creds = get_ga4_credentials()
    ga4_client = BetaAnalyticsDataClient(credentials=ga4_creds)

    # Every OLJ table in the report is filtered to OLJ streams (name contains "olj",
        # case-insensitive, so it matches "OLJ Website" / "olj android" / "OLJ iOS" etc.)
    olj_stream_filter = FilterExpression(
        filter=Filter(
            field_name="streamName",
            string_filter=Filter.StringFilter(
                match_type=Filter.StringFilter.MatchType.CONTAINS,
                value="olj",
                case_sensitive=False,
            )
        )
    )


    def ga4_report(start, end, metrics, dimensions=None, order_by_metric=None, limit=None,
                   extra_filter=None, stream_filter=None):
        base_filter = stream_filter if stream_filter is not None else olj_stream_filter
        dim_filter = base_filter
        if extra_filter is not None:
            from google.analytics.data_v1beta.types import FilterExpressionList
            dim_filter = FilterExpression(
                and_group=FilterExpressionList(expressions=[base_filter, extra_filter])
            )
        request = RunReportRequest(
            property=f"properties/{GA4_PROPERTY_ID}",
            date_ranges=[DateRange(start_date=start.strftime("%Y-%m-%d"), end_date=end.strftime("%Y-%m-%d"))],
            metrics=[Metric(name=m) for m in metrics],
            dimensions=[Dimension(name=d) for d in (dimensions or [])],
            dimension_filter=dim_filter,
            limit=limit,
        )
        if order_by_metric:
            request.order_bys = [OrderBy(metric=OrderBy.MetricOrderBy(metric_name=order_by_metric), desc=True)]
        resp = ga4_client.run_report(request)
        cols = [d.name for d in resp.dimension_headers] + [m.name for m in resp.metric_headers]
        rows = []
        for row in resp.rows:
            rows.append([v.value for v in row.dimension_values] + [v.value for v in row.metric_values])
        return pd.DataFrame(rows, columns=cols)

    print(f"GA4 ready -- property {GA4_PROPERTY_ID}")
else:
    print("[GA4 not connected yet -- using example data for this section. "
          "Set GA4_PROPERTY_ID and make sure service_account.json is present + authorized on the property.]")


[GA4 not connected yet -- using example data for this section. Set GA4_PROPERTY_ID and make sure service_account.json is present + authorized on the property.]


### 3a. Scorecard metrics — Users, Sessions, Page views (Web / App / Total)

**Total is pulled as its own query, not Web + App**: `totalUsers` dedupes people who used both the site
and the app, so Web + App users will be slightly *higher* than Total — that's expected, not a bug.

In [ ]:
# --- Web vs App split (used by every GA4 table below) ---
# GA4's `platform` dimension is "web", "iOS" or "Android" -> web = Web, iOS + Android = App.
PLATFORMS = ["Web", "App"]
SCORE_METRICS = {"users": "totalUsers", "sessions": "sessions", "page_views": "screenPageViews"}

def platform_bucket(p):
    return "Web" if str(p).strip().lower() == "web" else "App"

def ga4_scorecard(start, end, stream_filter=None):
    """{'Web': {...}, 'App': {...}, 'Total': {...}}, each with users / sessions / page_views."""
    if GA4_READY:
        mets = list(SCORE_METRICS.values())
        split = ga4_report(start, end, metrics=mets, dimensions=["platform"], stream_filter=stream_filter)
        split[mets] = split[mets].astype(int)
        by_bucket = split.assign(bucket=split["platform"].map(platform_bucket)).groupby("bucket")[mets].sum()
        total_row = ga4_report(start, end, metrics=mets, stream_filter=stream_filter).iloc[0]  # deduped total
        out = {b: {k: int(by_bucket.at[b, m]) if b in by_bucket.index else 0 for k, m in SCORE_METRICS.items()}
               for b in PLATFORMS}
        out["Total"] = {k: int(total_row[m]) for k, m in SCORE_METRICS.items()}
        return out
    else:
        # Example data
        base = 42000 if start == WEEK_START else 39900
        web_share = 0.64 if start == WEEK_START else 0.61
        total = {"users": base, "sessions": int(base * 1.39), "page_views": int(base * 2.66)}
        web = {k: int(v * web_share) for k, v in total.items()}
        app = {k: v - web[k] for k, v in total.items()}
        app["users"] = int(app["users"] * 1.06)  # some web/app overlap -> Web + App users > Total
        return {"Web": web, "App": app, "Total": total}

this_week_ga4 = ga4_scorecard(WEEK_START, WEEK_END)
last_week_ga4 = ga4_scorecard(PREV_WEEK_START, PREV_WEEK_END)

def wow_pct(this_v, last_v):
    if last_v == 0:
        return "n/a"
    pct = (this_v - last_v) / last_v * 100
    arrow = "\u25b2" if pct >= 0 else "\u25bc"
    return f"{arrow} {abs(pct):.0f}%"

def signed_wow_pct(this_v, last_v):
    """For metrics that can be negative (like Net Subscription Change): direction is
    based on whether the value actually improved (this_v >= last_v), not the raw sign of
    the % change -- plain division misleads here, since -27 -> -40 is worse but naive
    math (dividing two negatives) would show it as a positive-looking \"+48%\"."""
    if last_v == 0:
        return "n/a"
    pct = abs(this_v - last_v) / abs(last_v) * 100
    arrow = "\u25b2" if this_v >= last_v else "\u25bc"
    return f"{arrow} {pct:.0f}%"

def pct_change(this_v, last_v):
    """Raw % change (None when last week was 0) -- the report colors arrows from this number."""
    if not last_v:
        return None
    return (this_v - last_v) / last_v * 100

GA4_ROWS = [("users", "Users"), ("sessions", "Sessions"), ("page_views", "Page views")]

scorecard_df = pd.DataFrame([
    {"Metric": label,
     **{f"{g} {col}": val
        for g in ["Total"] + PLATFORMS
        for col, val in zip(["last", "this", "WoW"],
                            [last_week_ga4[g][k], this_week_ga4[g][k],
                             wow_pct(this_week_ga4[g][k], last_week_ga4[g][k])])}}
    for k, label in GA4_ROWS
]).set_index("Metric")

scorecard_df

### 3b. Top 10 articles

In [9]:
def ga4_top_articles(start, end, n=10, stream_filter=None):
    if GA4_READY:
        df = ga4_report(start, end, metrics=["screenPageViews"], dimensions=["pageTitle"],
                         order_by_metric="screenPageViews", limit=n, stream_filter=stream_filter)
        df["screenPageViews"] = df["screenPageViews"].astype(int)
        return df.rename(columns={"pageTitle": "Article", "screenPageViews": "Views"})
    else:
        return pd.DataFrame({
            "Article": [f"Example article title {i}" for i in range(1, n + 1)],
            "Views": sorted([4200, 3100, 2650, 2200, 1800, 1600, 1400, 1250, 1100, 950], reverse=True)[:n],
        })

top_articles = ga4_top_articles(WEEK_START, WEEK_END)
top_articles


,Article,Views
0,Example article title 1,4200
1,Example article title 2,3100
2,Example article title 3,2650
3,Example article title 4,2200
4,Example article title 5,1800
5,Example article title 6,1600
6,Example article title 7,1400
7,Example article title 8,1250
8,Example article title 9,1100
9,Example article title 10,950


### 3e. Top 10 articles (web + app views summed by Article ID, with a Web / App split)

Web events write the article ID to the `articleid` custom dimension; app events write it to a
**separate** `article_id` dimension — so this pulls both independently (each already filtered to OLJ
streams) and merges them by matching ID value, summing views across web + app for the same article
before ranking. `platform` is pulled in the same query, so each article also gets its
Web and App views as separate columns (ranking is still by the total).

Also pulls `sessionSource` in the same query, so each article gets its **top 2 traffic sources**
shown as a percentage of that article's own views (e.g. "google 54% · (direct) 22%") — not a
share of site-wide traffic.

In [ ]:
import re

ARTICLE_ID_PATTERN = re.compile(r"^\d{6,7}$")  # valid article IDs only -- drops "(not set)" and junk

def _top2_sources(rows):
    """rows: list of (source, views) for one article. Returns two 'source NN%' strings, where the
    percentage is that source's share of THIS article's views (not overall site traffic)."""
    total = sum(v for _, v in rows)
    if total == 0:
        return "", ""
    ranked = sorted(rows, key=lambda t: t[1], reverse=True)[:2]
    out = [f"{src} {round(v / total * 100)}%" for src, v in ranked]
    while len(out) < 2:
        out.append("")
    return out[0], out[1]

def ga4_top_articles_by_id(start, end, n=10, stream_filter=None):
    if GA4_READY:
        # pageTitle + sessionSource pulled in the SAME query as the ID -- not a separate lookup call
        web = ga4_report(start, end, metrics=["screenPageViews"],
                          dimensions=["customEvent:articleid", "sessionSource", "pageTitle", "platform"], stream_filter=stream_filter)
        app = ga4_report(start, end, metrics=["screenPageViews"],
                          dimensions=["customEvent:article_id", "sessionSource", "pageTitle", "platform"], stream_filter=stream_filter)

        web = web.rename(columns={"customEvent:articleid": "article_id", "screenPageViews": "views"})
        app = app.rename(columns={"customEvent:article_id": "article_id", "screenPageViews": "views"})

        combined = pd.concat([web, app], ignore_index=True)
        combined["views"] = combined["views"].astype(int)

        # Keep only valid numeric article IDs (6 or 7 digits) -- excludes "(not set)", blanks, junk
        valid = combined["article_id"].astype(str).str.strip().apply(lambda v: bool(ARTICLE_ID_PATTERN.match(v)))
        combined = combined[valid]

        # Views per article split by platform (web -> Web, iOS/Android -> App), ranked by the total
        combined["bucket"] = combined["platform"].map(platform_bucket)
        totals = combined.pivot_table(index="article_id", columns="bucket", values="views",
                                      aggfunc="sum", fill_value=0).reindex(columns=PLATFORMS, fill_value=0)
        totals.columns.name = None
        totals["views"] = totals["Web"] + totals["App"]
        top = totals.sort_values("views", ascending=False).head(n).reset_index()

        # Rank by ID first (above), THEN attach a title per ID -- the most frequent title seen for
        # that ID across both web and app rows (guards against a stray differently-formatted title)
        subset = combined[combined["article_id"].isin(top["article_id"])]
        titles = (
            subset.groupby("article_id")["pageTitle"]
            .agg(lambda s: s.value_counts().idxmax() if len(s) else "")
            .to_dict()
        )

        # Top 2 traffic sources per article, as a share of that article's own views
        source_totals = subset.groupby(["article_id", "sessionSource"])["views"].sum()
        src1, src2 = {}, {}
        for aid in top["article_id"]:
            rows = list(source_totals.loc[aid].items()) if aid in source_totals.index.get_level_values(0) else []
            s1, s2 = _top2_sources(rows)
            src1[aid], src2[aid] = s1, s2

        top["Article"] = top["article_id"].map(titles)
        top["Top source 1"] = top["article_id"].map(src1)
        top["Top source 2"] = top["article_id"].map(src2)
        return top.rename(columns={"article_id": "Article ID", "views": "Views"})[
            ["Article", "Article ID", "Web", "App", "Views", "Top source 1", "Top source 2"]
        ]
    else:
        example_sources = [
            ("google 54%", "(direct) 22%"), ("facebook 41%", "google 30%"),
            ("(direct) 38%", "newsletter 19%"), ("google 47%", "instagram 15%"),
            ("(direct) 33%", "google 28%"), ("google 39%", "bing 12%"),
            ("newsletter 44%", "(direct) 21%"), ("google 36%", "facebook 20%"),
            ("(direct) 29%", "google 24%"), ("google 31%", "(direct) 26%"),
        ][:n]
        views = sorted([4200, 3100, 2650, 2200, 1800, 1600, 1400, 1250, 1100, 950], reverse=True)[:n]
        web = [int(v * s) for v, s in zip(views, [0.62, 0.55, 0.70, 0.48, 0.66, 0.59, 0.73, 0.51, 0.64, 0.57])]
        return pd.DataFrame({
            "Article": [f"Example article title {i}" for i in range(1, n + 1)],
            "Article ID": [f"15481{i:02d}" for i in range(1, n + 1)],
            "Web": web,
            "App": [v - w for v, w in zip(views, web)],
            "Views": views,
            "Top source 1": [s[0] for s in example_sources],
            "Top source 2": [s[1] for s in example_sources],
        })

top_articles_by_id = ga4_top_articles_by_id(WEEK_START, WEEK_END)
top_articles_by_id

### 3c. Top countries & top sources

Countries (and source categories in 3f) are pulled as `<dimension> × platform`, so each row gets a
**Web / App / Total** share — i.e. "Lebanon = 48% of web sessions, 41% of app sessions, 45% overall".
Shares are of **all** sessions for that platform, not just of the top 5.

In [ ]:
def ga4_top_dimension(start, end, dimension, n=5, stream_filter=None):
    if GA4_READY:
        df = ga4_report(start, end, metrics=["sessions"], dimensions=[dimension],
                         order_by_metric="sessions", limit=n, stream_filter=stream_filter)
        df["sessions"] = df["sessions"].astype(int)
        total = df["sessions"].sum()
        df["share"] = (df["sessions"] / total * 100).round(0).astype(int).astype(str) + "%"
        return df.rename(columns={dimension: dimension.capitalize(), "sessions": "Sessions", "share": "Share"})
    else:
        # Example data varies slightly by week so the WoW comparison below has something to show
        if dimension == "country":
            if start == WEEK_START:
                data = {"Country": ["Lebanon", "France", "USA", "Canada", "UAE"],
                        "Sessions": [24800, 8900, 6200, 3100, 2400]}
            else:
                data = {"Country": ["Lebanon", "France", "USA", "UAE", "Canada"],
                        "Sessions": [23100, 9400, 5800, 2600, 2200]}
            df = pd.DataFrame(data).head(n)
            df["Share"] = (df["Sessions"] / df["Sessions"].sum() * 100).round(0).astype(int).astype(str) + "%"
            return df
        else:
            df = pd.DataFrame({
                "Sessionsource": ["(direct)", "google", "olj", "website", "bing", "CMS-9", "(not set)"],
                "Sessions": [15200, 11800, 9400, 5900, 3100, 1900, 1200],
            }).head(n)
            df["Share"] = (df["Sessions"] / df["Sessions"].sum() * 100).round(0).astype(int).astype(str) + "%"
            return df

def ga4_split_by(start, end, dimension, metric="sessions", stream_filter=None):
    """<dimension> x platform, pivoted to one row per value with Web / App / Total counts.
    Sessions add up cleanly across platforms, so Total = Web + App here."""
    df = ga4_report(start, end, metrics=[metric], dimensions=[dimension, "platform"], limit=100000,
                    stream_filter=stream_filter)
    if df.empty:
        return pd.DataFrame(columns=PLATFORMS + ["Total"], dtype=int)
    df[metric] = df[metric].astype(int)
    df["bucket"] = df["platform"].map(platform_bucket)
    out = df.pivot_table(index=dimension, columns="bucket", values=metric, aggfunc="sum", fill_value=0)
    out = out.reindex(columns=PLATFORMS, fill_value=0)
    out.columns.name = None
    out["Total"] = out["Web"] + out["App"]
    return out.sort_values("Total", ascending=False)

def _share(num, den):
    return f"{num / den * 100:.0f}%" if den else "—"

def split_share_table(this_counts, last_counts, labels, label_col):
    """One row per label. For each of Total / Web / App: '<g> last' and '<g> this' = the label's share
    of that platform's sessions, '<g> WoW' = % change in the label's session COUNT (not the share)."""
    groups = ["Total"] + PLATFORMS
    this_tot = this_counts[groups].sum()
    last_tot = last_counts[groups].sum()
    rows = []
    for lab in labels:
        t = this_counts.loc[lab, groups] if lab in this_counts.index else pd.Series(0, index=groups)
        l = last_counts.loc[lab, groups] if lab in last_counts.index else None
        row = {label_col: lab}
        for g in groups:
            row[f"{g} last"] = _share(l[g], last_tot[g]) if l is not None else "—"
            row[f"{g} this"] = _share(t[g], this_tot[g])
            row[f"{g} WoW"] = pct_change(int(t[g]), int(l[g])) if l is not None else None
        rows.append(row)
    return pd.DataFrame(rows)

def country_counts(start, end):
    if GA4_READY:
        return ga4_split_by(start, end, "country", stream_filter=olj_stream_filter)
    # Example data (Web, App sessions)
    if start == WEEK_START:
        data = {"Lebanon": (14800, 10000), "France": (6600, 2300), "USA": (4300, 1900),
                "Canada": (2000, 1100), "UAE": (1300, 1100), "Germany": (900, 300), "Belgium": (700, 200)}
    else:
        data = {"Lebanon": (13600, 9500), "France": (7000, 2400), "USA": (4000, 1800),
                "UAE": (1500, 1100), "Canada": (1500, 700), "Germany": (850, 300), "Belgium": (650, 200)}
    df = pd.DataFrame.from_dict(data, orient="index", columns=PLATFORMS)
    df["Total"] = df["Web"] + df["App"]
    return df.sort_values("Total", ascending=False)

this_country_counts = country_counts(WEEK_START, WEEK_END)
last_country_counts = country_counts(PREV_WEEK_START, PREV_WEEK_END)
top_countries = split_share_table(this_country_counts, last_country_counts,
                                  list(this_country_counts.head(5).index), "Country")
top_sources = ga4_top_dimension(WEEK_START, WEEK_END, "sessionSource", stream_filter=olj_stream_filter)

print("Top countries (with last week)")
display(top_countries)
print("\nTop sources")
display(top_sources)

### 3f. Sources mapped into categories (OLJ) — Web / App / Total

In [ ]:
MAIN_CATS = ["Direct", "Search Engines", "Social Networks", "AI Assistants", "Internal/Newsletters", "Other"]

MAIN_COLORS = {
    "Direct":               "4285F4",
    "Search Engines":       "00BFA5",
    "Social Networks":      "9C27B0",
    "AI Assistants":        "FFA726",
    "Internal/Newsletters": "FF6B9D",
    "Other":                "9E9E9E",
}

DIRECT = {"(direct)"}

SEARCH_ENGINES = {
    "google", "news.google.com", "bing", "ecosia.org", "qwant.com", "duckduckgo",
    "fr.search.yahoo.com", "yahoo", "search.brave.com", "yandex", "ya.ru", "startpage.com",
}

AI_ASSISTANTS = {
    "chatgpt.com", "perplexity.ai", "gemini.google.com", "perplexity", "copilot.com",
    "copilot.microsoft.com", "openai", "duck.ai", "chat.mistral.ai",
    "notebooklm.google.com", "claude.ai", "poe.com", "grok.com", "chat.qwen.ai",
    "doubao.com", "chat.z.ai", "felo.ai", "mammouth.ai", "you.com",
}

# --- OLJ-specific sets ---
SOCIAL_NETWORKS_OLJ = {
    "m.facebook.com", "facebook.com", "l.facebook.com", "lm.facebook.com",
    "mobile.facebook.com", "facebook", "l.instagram.com", "ig", "instagram",
    "instagram.com", "later-linkinbio", "linkin.bio", "t.co", "linkedin.com",
    "lnkd.in", "go.bsky.app", "l.threads.com", "twitter", "x.com", "threads",
    "bluesky", "pinterest.com", "tiktok.com", "snapchat", "snapchat.com",
    "web.whatsapp.com", "cms-46", "reddit.com", "fb", "flipboard", "flipboard.com",
}

INTERNAL_NEWSLETTERS_OLJ = {
    "mailchimp", "newsletter", "email", "website", "olj", "google-play", "olj.me",
    "olj.wael", "autopromoolj", "morningbrief", "hs_email", "brevo", "mailchi.mp",
    "us1.campaign-archive.com", "actito.be", "activetrail", "acumbamail", "omnisend",
    "wordfly", "newsletter_1", "newsletter_6b", "newsletter_paiementechouepp",
    "newsletter_preventif", "partenairesjamhour", "marketo",
    "gmi mailchimp integration prod list", "master list", "bundle", "nb",
}


def categorize_source(src, social=SOCIAL_NETWORKS_OLJ, newsletters=INTERNAL_NEWSLETTERS_OLJ):
    s = str(src).strip().lower()
    if s in DIRECT:
        return "Direct"
    if s in SEARCH_ENGINES:
        return "Search Engines"
    if s in social:
        return "Social Networks"
    if s in AI_ASSISTANTS:
        return "AI Assistants"
    if s in newsletters:
        return "Internal/Newsletters"
    return "Other"

def ga4_sources_by_category(start, end, stream_filter=None, social=SOCIAL_NETWORKS_OLJ, newsletters=INTERNAL_NEWSLETTERS_OLJ):
    """Sessions per source category, split Web / App / Total (index = MAIN_CATS)."""
    if GA4_READY:
        # Every distinct source (no top-N slice) so categorization is complete
        by_src = ga4_split_by(start, end, "sessionSource", stream_filter=stream_filter)
        cats = by_src.index.map(lambda s: categorize_source(s, social, newsletters))
        return by_src.groupby(cats).sum().reindex(MAIN_CATS, fill_value=0)
    # Example data (Web, App sessions) -- varies by week so WoW has something to show
    if start == WEEK_START:
        data = {"Direct": (8400, 6800), "Search Engines": (11900, 1800), "Social Networks": (6900, 2500),
                "AI Assistants": (1100, 100), "Internal/Newsletters": (5600, 3800), "Other": (2300, 800)}
    else:
        data = {"Direct": (7900, 6200), "Search Engines": (10700, 1600), "Social Networks": (7600, 2600),
                "AI Assistants": (820, 80), "Internal/Newsletters": (5200, 3500), "Other": (2100, 800)}
    df = pd.DataFrame.from_dict(data, orient="index", columns=PLATFORMS).reindex(MAIN_CATS, fill_value=0)
    df["Total"] = df["Web"] + df["App"]
    return df

def sources_by_category_with_wow(stream_filter=None):
    this_df = ga4_sources_by_category(WEEK_START, WEEK_END, stream_filter=stream_filter)
    last_df = ga4_sources_by_category(PREV_WEEK_START, PREV_WEEK_END, stream_filter=stream_filter)
    out = split_share_table(this_df, last_df, MAIN_CATS, "Category")
    out["Color"] = out["Category"].map(MAIN_COLORS)
    return out

sources_by_category = sources_by_category_with_wow(stream_filter=olj_stream_filter)
sources_by_category

### 3d. App downloads — iOS vs Android

In [13]:
def ga4_app_downloads(start, end, stream_filter=None):
    if GA4_READY:
        from google.analytics.data_v1beta.types import Filter as _Filter, FilterExpression as _FE
        event_filter = _FE(filter=_Filter(field_name="eventName",
                                           string_filter=_Filter.StringFilter(value="app_download")))
        df = ga4_report(start, end, metrics=["eventCount"], dimensions=["platform"],
                         extra_filter=event_filter, stream_filter=stream_filter)
        counts = {row["platform"]: int(row["eventCount"]) for _, row in df.iterrows()}
        return {"ios": counts.get("iOS", 0), "android": counts.get("Android", 0)}
    else:
        return {"ios": 520, "android": 370} if start == WEEK_START else {"ios": 480, "android": 337}

this_week_downloads = ga4_app_downloads(WEEK_START, WEEK_END)
last_week_downloads = ga4_app_downloads(PREV_WEEK_START, PREV_WEEK_END)

downloads_df = pd.DataFrame([
    {"Metric": "App downloads (iOS / Android)",
     "This week": f"{sum(this_week_downloads.values())} ({this_week_downloads['ios']} / {this_week_downloads['android']})",
     "Last week": f"{sum(last_week_downloads.values())} ({last_week_downloads['ios']} / {last_week_downloads['android']})",
     "WoW": wow_pct(sum(this_week_downloads.values()), sum(last_week_downloads.values()))}
]).set_index("Metric")

downloads_df


,This week,Last week,WoW
Metric,,,
App downloads (iOS / Android),890 (520 / 370),817 (480 / 337),▲ 9%


## 4. Final report — OLJ Weekly Brief (preview)

Builds the numbers into the brief once (`REPORT`) and shows a **color-coded preview right here** — the
Word export in section 5 is drawn from the exact same data and colors, so what you see is what gets sent.

Layout, designed to fit on **one A4 page**:
- Every comparison table has three color blocks, each with **Last wk · This wk · WoW**:
  **WEB** (blue) · **APP** (green) · **TOTAL** (charcoal, bold, last).
  Reading just the Total block gives the whole picture; Web/App are there when you want the why.
- WoW arrows are green ▲ / red ▼. Top-articles titles are clipped so each row stays on one line.

The report opens straight on the scorecard. The only commentary is "one thing to watch" at the very end,
auto-flagged from any metric down 10%+ WoW — a starting point, not a substitute for reading the numbers.

In [ ]:
from IPython.display import HTML, display
import html as _html

# Fail with a clear message (instead of a NameError) if an earlier section wasn't run in this session
_NEEDED = {
    "this_week": "1. Acquisitions & Churn",
    "this_week_accounts": "2. New accounts",
    "ACQUISITIONS_SOURCE": "2b. New subscriptions",
    "this_week_ga4": "3a. Scorecard",
    "top_articles_by_id": "3e. Top 10 articles",
    "top_countries": "3c. Top countries",
    "sources_by_category": "3f. Sources by category",
    "this_week_downloads": "3d. App downloads",
}
_missing = [sec for var, sec in _NEEDED.items() if var not in globals()]
if _missing:
    raise RuntimeError("These sections haven't been run in this session -- run them (or Runtime > Run all) "
                       "and then this cell again:\n  - " + "\n  - ".join(_missing))

tw, lw = this_week_ga4, last_week_ga4
dl_this_total = sum(this_week_downloads.values())
dl_last_total = sum(last_week_downloads.values())

# ---------- "One thing to watch" (from the TOTALS; Web/App only when they tell a different story) ----------
movers = {label: pct_change(tw["Total"][k], lw["Total"][k]) for k, label in GA4_ROWS}
movers["App downloads"] = pct_change(dl_this_total, dl_last_total)

watch_lines = []
for name, pct in movers.items():
    if pct is not None and pct <= -10:
        watch_lines.append(f"{name} down {abs(pct):.0f}% week-over-week.")
for k, label in GA4_ROWS:  # one platform sliding while the other hides it in the total
    if movers[label] is None or movers[label] > -10:
        for p in PLATFORMS:
            pc = pct_change(tw[p][k], lw[p][k])
            if pc is not None and pc <= -10:
                watch_lines.append(f"{p} {label.lower()} down {abs(pc):.0f}% week-over-week.")
new_subs_delta = this_week["olj_new"] - last_week["olj_new"]
if new_subs_delta < 0:
    watch_lines.append(f"New OLJ subscriptions down {abs(new_subs_delta)} vs. last week ({this_week['olj_new']} this week).")
watch_line = " ".join(watch_lines) if watch_lines else "Nothing off-trend this week."

# ---------- Look & feel (shared by this preview AND the Word export) ----------
GROUP_ORDER = ["Web", "App", "Total"]  # Total last, so it reads as Web + App = Total
GROUP_STYLE = {                          # head = block header, sub = sub-header, cell = body tint, text = accent
    "Total": {"head": "2B3440", "sub": "DDE1E6", "cell": "F3F4F6", "text": "2B3440"},
    "Web":   {"head": "1A73E8", "sub": "D6E4FB", "cell": "F0F5FE", "text": "1A5BB8"},
    "App":   {"head": "1E8E3E", "sub": "D3ECD9", "cell": "F0F8F2", "text": "17702F"},
}
INK, SOFT, MUTED = "2B3440", "5F6368", "8A9096"
UP_COLOR, DOWN_COLOR = "137333", "C5221F"
WATCH_ACCENT, WATCH_FILL = "E37400", "FEF4E6"
SUB_HEADERS = ["Last wk", "This wk", "WoW"]
TITLE_CLIP = 80

WEEK_LINE = (f"Week of {WEEK_START:%b %d} – {WEEK_END:%b %d, %Y}   ·   "
             f"compared with {PREV_WEEK_START:%b %d} – {PREV_WEEK_END:%b %d}")
FOOTNOTE = ("Web = website · App = iOS + Android · Total users are de-duplicated, so Web + App users can exceed Total · "
            "Country / source figures are each platform's share of its own sessions; WoW is the change in sessions · "
            "New accounts: Web / App from the CMS acquisition source; 'other' (newsletters etc.) counts in Total only · "
            "New subscriptions: paid subscriptions of new accounts; App = Apple / Google Play; free (staff) excluded.")

def fmt_wow(p):
    if p is None or pd.isna(p):
        return "n/a"
    return f"{'▲' if p >= 0 else '▼'} {abs(p):.0f}%"

def wow_color(p):
    if p is None or pd.isna(p):
        return MUTED
    return UP_COLOR if p >= 0 else DOWN_COLOR

def clip(s, n=TITLE_CLIP):
    s = str(s)
    return s if len(s) <= n else s[: n - 1].rstrip() + "…"

def cmp_counts(this_v, last_v):
    return (f"{last_v:,}", f"{this_v:,}", pct_change(this_v, last_v))

def share_rows(df, label_col):
    return [{"label": r[label_col], **{g: (r[f"{g} last"], r[f"{g} this"], r[f"{g} WoW"]) for g in GROUP_ORDER}}
            for _, r in df.iterrows()]

def new_accounts_row(t, l):
    """Web / App split when the CMS supplied one; 'Other' (newsletters etc.) is in Total and noted under the label."""
    row = {"label": "New accounts", "Web": None, "App": None,
           "Total": cmp_counts(t["olj_new_accounts"], l["olj_new_accounts"])}
    ts, ls = t.get("olj_split"), l.get("olj_split")
    if ts and ls:
        row["Web"] = cmp_counts(ts["Web"], ls["Web"])
        row["App"] = cmp_counts(ts["App"], ls["App"])
        if ts["Other"] or ls["Other"]:
            row["note"] = f"incl. {ts['Other']:,} other · {ls['Other']:,} last wk"
    return row

def new_subs_row(t, l):
    """Web / App split when section 2b supplied one (Free / staff and test payments already excluded)."""
    row = {"label": "New OLJ subscriptions", "Web": None, "App": None,
           "Total": cmp_counts(t["olj_new"], l["olj_new"])}
    ts, ls = t.get("olj_new_split"), l.get("olj_new_split")
    if ts and ls:
        row["Web"] = cmp_counts(ts["Web"], ls["Web"])
        row["App"] = cmp_counts(ts["App"], ls["App"])
    return row

# ---------- The report data: one row = {"label", "note"?, "Total"/"Web"/"App": (last, this, pct) or None} ----------
scorecard_rows = [
    {"label": label, **{g: cmp_counts(tw[g][k], lw[g][k]) for g in GROUP_ORDER}} for k, label in GA4_ROWS
] + [
    new_accounts_row(this_week_accounts, last_week_accounts),
    new_subs_row(this_week, last_week),
    {"label": "App downloads", "note": f"iOS {this_week_downloads['ios']} · Android {this_week_downloads['android']}",
     "Web": None, "Total": cmp_counts(dl_this_total, dl_last_total), "App": cmp_counts(dl_this_total, dl_last_total)},
]

def _sources_cell(row):
    s = row["Top source 1"]
    if row["Top source 2"]:
        s += f" · {row['Top source 2']}"
    return s

article_rows = [
    {"rank": i + 1, "title": row["Article"], "Total": f"{row['Views']:,}", "Web": f"{row['Web']:,}",
     "App": f"{row['App']:,}", "sources": _sources_cell(row)}
    for i, row in top_articles_by_id.reset_index(drop=True).iterrows()
]

REPORT = {
    "scorecard": ("Metric", scorecard_rows),
    "countries": ("Country", share_rows(top_countries, "Country")),
    "sources": ("Source category", share_rows(sources_by_category, "Category")),
    "articles": article_rows,
}

# ---------- Notebook preview (HTML, same colors as the .docx) ----------
def _h(s):
    return _html.escape(str(s))

_SEP = "border-left:3px solid #fff;"
def _td(content, style=""):
    return f'<td style="padding:3px 7px;border-bottom:1px solid #E6E8EB;white-space:nowrap;{style}">{content}</td>'

def _html_group_cells(cell, g):
    st = GROUP_STYLE[g]
    base = f"background:#{st['cell']};text-align:center;"
    if cell is None:
        return "".join(_td("—", base + f"color:#{MUTED};" + (_SEP if i == 0 else "")) for i in range(3))
    last, this, p = cell
    tot = g == "Total"
    return (_td(_h(last), base + _SEP + f"color:#{SOFT};font-weight:{600 if tot else 400};")
            + _td(_h(this), base + f"color:#{INK};font-weight:{800 if tot else 600};")
            + _td(fmt_wow(p), base + f"color:#{wow_color(p)};font-weight:700;"))

def html_cmp_table(label_header, rows):
    top = (f'<th rowspan="2" style="text-align:left;padding:4px 8px;color:#{INK};'
           f'border-bottom:2px solid #{INK}">{_h(label_header)}</th>')
    sub = ""
    for g in GROUP_ORDER:
        st = GROUP_STYLE[g]
        top += (f'<th colspan="3" style="background:#{st["head"]};color:#fff;padding:4px;font-size:11px;'
                f'letter-spacing:.08em;{_SEP}">{g.upper()}</th>')
        sub += "".join(f'<th style="background:#{st["sub"]};color:#{st["text"]};padding:3px 7px;font-size:10.5px;'
                       f'{_SEP if i == 0 else ""}">{s}</th>' for i, s in enumerate(SUB_HEADERS))
    body = ""
    for r in rows:
        label = _h(r["label"]) + (f'<div style="color:#{MUTED};font-size:10px;font-weight:400">{_h(r["note"])}</div>'
                                   if r.get("note") else "")
        body += "<tr>" + _td(label, f"text-align:left;color:#{INK};font-weight:600;") + "".join(
            _html_group_cells(r.get(g), g) for g in GROUP_ORDER) + "</tr>"
    return (f'<table style="border-collapse:collapse;width:100%;font-size:12px">'
            f'<tr>{top}</tr><tr>{sub}</tr>{body}</table>')

def html_articles_table(rows):
    neutral = f'text-align:left;padding:4px 8px;color:#{INK};border-bottom:2px solid #{INK};'
    head = (f'<th style="{neutral}">#</th><th style="{neutral}">Article</th>'
            + "".join(f'<th style="background:#{GROUP_STYLE[g]["head"]};color:#fff;padding:4px 8px;font-size:11px;'
                      f'letter-spacing:.08em;{_SEP}">{g.upper()}</th>' for g in GROUP_ORDER)
            + f'<th style="{neutral}{_SEP}">Top sources</th>')
    body = ""
    for r in rows:
        cells = _td(r["rank"], f"color:#{MUTED};") + _td(
            f'<span title="{_h(r["title"])}">{_h(clip(r["title"]))}</span>', f"color:#{INK};white-space:normal;")
        for g in GROUP_ORDER:
            cells += _td(r[g], f"background:#{GROUP_STYLE[g]['cell']};text-align:center;{_SEP}"
                              f"color:#{INK};font-weight:{800 if g == 'Total' else 500};")
        cells += _td(_h(r["sources"]), f"color:#{SOFT};font-size:11px;{_SEP}")
        body += f"<tr>{cells}</tr>"
    return f'<table style="border-collapse:collapse;width:100%;font-size:12px"><tr>{head}</tr>{body}</table>'

def _html_section(title, inner):
    return (f'<div style="font-size:11px;font-weight:700;letter-spacing:.1em;color:#{INK};margin:16px 0 6px">'
            f'{_h(title.upper())}</div>{inner}')

def _html_callout(label, text, accent, fill):
    return (f'<div style="background:#{fill};border-left:4px solid #{accent};padding:8px 12px;margin-top:12px;'
            f'border-radius:4px"><div style="font-size:10px;font-weight:700;letter-spacing:.1em;color:#{accent}">'
            f'{_h(label.upper())}</div><div style="font-size:13px;font-weight:600;color:#{INK};margin-top:2px">'
            f'{_h(text)}</div></div>')

def html_report():
    return (
        f'<div style="font-family:Calibri,Carlito,\'Segoe UI\',Arial,sans-serif;color:#{INK};background:#fff;'
        f'max-width:940px;padding:22px 26px;border:1px solid #E6E8EB;border-radius:10px">'
        f'<div style="font-size:24px;font-weight:800">Weekly Analytics Brief <span style="color:#{MUTED};'
        f'font-weight:600">· OLJ</span></div>'
        f'<div style="font-size:12px;color:#{SOFT};margin-top:2px">{_h(WEEK_LINE)}</div>'
        + _html_section("Scorecard", html_cmp_table(*REPORT["scorecard"]))
        + _html_section("Top 10 articles this week", html_articles_table(REPORT["articles"]))
        + _html_section("Top countries", html_cmp_table(*REPORT["countries"]))
        + _html_section("Sources by category", html_cmp_table(*REPORT["sources"]))
        + _html_callout("One thing to watch", watch_line, WATCH_ACCENT, WATCH_FILL)
        + f'<div style="font-size:10px;color:#{MUTED};margin-top:10px">{_h(FOOTNOTE)}</div></div>'
    )

display(HTML(html_report()))

## 5. Export (check it before anything is sent)

Builds the **one-page A4 `.docx`** from the same `REPORT` data and colors as the preview above, plus a
**PDF** when LibreOffice is available (optional — in Colab: `!apt-get -qq install -y libreoffice-writer`).
In Colab both files are offered as downloads so you can open them and check. **Nothing is emailed here** —
that's section 6.

In [ ]:
import shutil
import subprocess

from docx import Document
from docx.enum.table import WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Cm, Pt, RGBColor

PAGE_W, PAGE_H = 21.0, 29.7          # A4 portrait, cm
MARGIN_X, MARGIN_TOP, MARGIN_BOTTOM = 1.3, 1.1, 1.0
CONTENT_W = PAGE_W - 2 * MARGIN_X
BASE_PT = 8                          # body size; everything is sized so the brief fits on one page
DOWNLOAD_IN_COLAB = True

_ALIGN = {"left": WD_ALIGN_PARAGRAPH.LEFT, "center": WD_ALIGN_PARAGRAPH.CENTER}
_TCPR_ORDER = ["cnfStyle", "tcW", "gridSpan", "hMerge", "vMerge", "tcBorders", "shd", "noWrap", "tcMar",
               "textDirection", "tcFitText", "vAlign", "hideMark"]
_TBLPR_ORDER = ["tblStyle", "tblpPr", "tblOverlap", "bidiVisual", "tblStyleRowBandSize", "tblStyleColBandSize",
                "tblW", "jc", "tblCellSpacing", "tblInd", "tblBorders", "shd", "tblLayout", "tblCellMar", "tblLook"]

def _el(tag, **attrs):
    e = OxmlElement(tag)
    for k, v in attrs.items():
        e.set(qn(f"w:{k}"), str(v))
    return e

def _reorder(parent, order):
    """Word is strict about child order inside tcPr / tblPr -- sort what we appended into schema order."""
    kids = list(parent)
    rank = lambda e: order.index(e.tag.split("}")[1]) if e.tag.split("}")[1] in order else len(order)
    for k in kids:
        parent.remove(k)
    for k in sorted(kids, key=rank):
        parent.append(k)

def _shade(cell, fill):
    cell._tc.get_or_add_tcPr().append(_el("w:shd", val="clear", color="auto", fill=fill))

def _borders(cell, **edges):  # edge=(size in 1/8 pt, hex color)
    tcPr = cell._tc.get_or_add_tcPr()
    b = tcPr.find(qn("w:tcBorders"))
    if b is None:
        b = _el("w:tcBorders"); tcPr.append(b)
    for edge, (sz, color) in edges.items():
        b.append(_el(f"w:{edge}", val="single", sz=sz, space=0, color=color))

def _tight(p, before=0, after=0):
    pf = p.paragraph_format
    pf.space_before, pf.space_after, pf.line_spacing = Pt(before), Pt(after), 1.0

def _run(p, text, size=BASE_PT, bold=False, color=INK, italic=False):
    r = p.add_run(str(text))
    r.font.size, r.bold, r.italic = Pt(size), bold, italic
    r.font.color.rgb = RGBColor.from_string(color)
    return r

def _write(cell, text, *, size=BASE_PT, bold=False, color=INK, align="center", fill=None, note=None):
    cell.text = ""
    p = cell.paragraphs[0]
    _tight(p)
    p.alignment = _ALIGN[align]
    _run(p, text, size, bold, color)
    if note:
        _run(p, "", size).add_break()
        _run(p, note, size - 1.5, color=MUTED)
    if fill:
        _shade(cell, fill)
    cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER

def _table(doc, nrows, widths_cm, rules=True):
    t = doc.add_table(rows=nrows, cols=len(widths_cm))
    t.autofit = False
    tblPr = t._tbl.tblPr
    tblW = tblPr.find(qn("w:tblW"))
    if tblW is None:
        tblW = _el("w:tblW"); tblPr.append(tblW)
    tblW.set(qn("w:type"), "dxa"); tblW.set(qn("w:w"), str(int(sum(widths_cm) * 567)))
    b = _el("w:tblBorders")  # light horizontal rules only -- no grid
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        on = rules and edge in ("bottom", "insideH")
        b.append(_el(f"w:{edge}", val="single" if on else "nil", sz=4, space=0, color="E3E6EA"))
    tblPr.append(b)
    mar = _el("w:tblCellMar")
    for edge, w in (("top", 22), ("left", 70), ("bottom", 22), ("right", 70)):
        mar.append(_el(f"w:{edge}", w=w, type="dxa"))
    tblPr.append(mar)
    for i, w in enumerate(widths_cm):
        t.columns[i].width = Cm(w)
        for c in t.columns[i].cells:
            c.width = Cm(w)
    return t

WHITE_SEP = (18, "FFFFFF")  # white gap between the Total / Web / App blocks

def docx_cmp_table(doc, label_header, rows, label_w=3.9):
    val_w = (CONTENT_W - label_w) / 9
    t = _table(doc, 2 + len(rows), [label_w] + [val_w] * 9)
    lab = t.cell(0, 0).merge(t.cell(1, 0))
    _write(lab, label_header, bold=True, align="left")
    _borders(lab, bottom=(10, INK))
    for gi, g in enumerate(GROUP_ORDER):
        st, c0 = GROUP_STYLE[g], 1 + 3 * gi
        h = t.cell(0, c0).merge(t.cell(0, c0 + 2))
        _write(h, g.upper(), size=BASE_PT - 0.5, bold=True, color="FFFFFF", fill=st["head"])
        _borders(h, left=WHITE_SEP)
        for i, s in enumerate(SUB_HEADERS):
            c = t.cell(1, c0 + i)
            _write(c, s, size=BASE_PT - 1, bold=True, color=st["text"], fill=st["sub"])
            if i == 0:
                _borders(c, left=WHITE_SEP)
    for ri, r in enumerate(rows, start=2):
        _write(t.cell(ri, 0), r["label"], bold=True, align="left", note=r.get("note"))
        for gi, g in enumerate(GROUP_ORDER):
            st, c0, cell, tot = GROUP_STYLE[g], 1 + 3 * gi, r.get(g), g == "Total"
            if cell is None:
                vals = [("—", MUTED, False)] * 3
            else:
                last, this, p = cell
                vals = [(last, SOFT, tot), (this, INK, True), (fmt_wow(p), wow_color(p), True)]
            for i, (text, color, bold) in enumerate(vals):
                c = t.cell(ri, c0 + i)
                _write(c, text, bold=bold, color=color, fill=st["cell"],
                       size=BASE_PT + (0.5 if tot and i == 1 else 0))
                if i == 0:
                    _borders(c, left=WHITE_SEP)
    return t

def docx_articles_table(doc, rows):
    num_w = 1.45
    widths = [0.55, 7.35, num_w, num_w, num_w]
    widths.append(CONTENT_W - sum(widths))
    t = _table(doc, 1 + len(rows), widths)
    for i, h in enumerate(["#", "Article"]):
        _write(t.cell(0, i), h, bold=True, align="left")
        _borders(t.cell(0, i), bottom=(10, INK))
    for gi, g in enumerate(GROUP_ORDER):
        c = t.cell(0, 2 + gi)
        _write(c, g.upper(), size=BASE_PT - 0.5, bold=True, color="FFFFFF", fill=GROUP_STYLE[g]["head"])
        _borders(c, left=WHITE_SEP)
    _write(t.cell(0, 5), "Top sources", bold=True, align="left")
    _borders(t.cell(0, 5), bottom=(10, INK), left=WHITE_SEP)
    for ri, r in enumerate(rows, start=1):
        _write(t.cell(ri, 0), r["rank"], color=MUTED, align="left")
        _write(t.cell(ri, 1), clip(r["title"]), align="left")
        for gi, g in enumerate(GROUP_ORDER):
            c = t.cell(ri, 2 + gi)
            _write(c, r[g], bold=(g == "Total"), fill=GROUP_STYLE[g]["cell"])
            _borders(c, left=WHITE_SEP)
        _write(t.cell(ri, 5), r["sources"], size=BASE_PT - 1, color=SOFT, align="left")
        _borders(t.cell(ri, 5), left=WHITE_SEP)
    return t

def docx_callout(doc, label, text, accent, fill):
    t = _table(doc, 1, [CONTENT_W], rules=False)
    c = t.cell(0, 0)
    _shade(c, fill)
    _borders(c, left=(28, accent))
    p = c.paragraphs[0]
    _tight(p)
    _run(p, label.upper(), BASE_PT - 1, True, accent)
    p2 = c.add_paragraph()
    _tight(p2)
    _run(p2, text, BASE_PT + 1.5, True, INK)

def docx_section(doc, title):
    p = doc.add_paragraph()
    _tight(p, before=9, after=3)
    _run(p, title.upper(), BASE_PT + 0.5, True, INK)

def build_brief_docx(path):
    doc = Document()
    sec = doc.sections[0]
    sec.page_width, sec.page_height = Cm(PAGE_W), Cm(PAGE_H)
    sec.left_margin = sec.right_margin = Cm(MARGIN_X)
    sec.top_margin, sec.bottom_margin = Cm(MARGIN_TOP), Cm(MARGIN_BOTTOM)
    normal = doc.styles["Normal"]
    normal.font.name, normal.font.size = "Calibri", Pt(BASE_PT)
    normal.element.get_or_add_rPr().get_or_add_rFonts().set(qn("w:eastAsia"), "Calibri")

    p = doc.paragraphs[0] if doc.paragraphs else doc.add_paragraph()
    _tight(p)
    _run(p, "Weekly Analytics Brief", 17, True, INK)
    _run(p, "  ·  OLJ", 17, True, MUTED)
    p = doc.add_paragraph()
    _tight(p, after=6)
    _run(p, WEEK_LINE, BASE_PT + 1, color=SOFT)

    docx_section(doc, "Scorecard")
    docx_cmp_table(doc, *REPORT["scorecard"])
    docx_section(doc, "Top 10 articles this week")
    docx_articles_table(doc, REPORT["articles"])
    docx_section(doc, "Top countries")
    docx_cmp_table(doc, *REPORT["countries"])
    docx_section(doc, "Sources by category")
    docx_cmp_table(doc, *REPORT["sources"])
    spacer = doc.add_paragraph()
    _tight(spacer, after=4)
    docx_callout(doc, "One thing to watch", watch_line, WATCH_ACCENT, WATCH_FILL)
    p = doc.add_paragraph()
    _tight(p, before=5)
    _run(p, FOOTNOTE, BASE_PT - 1.5, color=MUTED)

    zoom = doc.settings.element.find(qn("w:zoom"))  # python-docx's template omits a required attribute
    if zoom is not None and zoom.get(qn("w:percent")) is None:
        zoom.set(qn("w:percent"), "100")

    body = doc.element.body
    for tcPr in body.iter(qn("w:tcPr")):
        _reorder(tcPr, _TCPR_ORDER)
    for tcb in body.iter(qn("w:tcBorders")):
        _reorder(tcb, ["top", "left", "bottom", "right", "insideH", "insideV"])
    for tblPr in body.iter(qn("w:tblPr")):
        _reorder(tblPr, _TBLPR_ORDER)
    doc.save(path)
    return path

def export_pdf(docx_path):
    """Optional PDF copy (handy for checking the one-page layout). Needs LibreOffice; skipped if absent."""
    soffice = shutil.which("soffice") or shutil.which("libreoffice")
    if not soffice:
        return None
    out_dir = os.path.dirname(os.path.abspath(docx_path))
    subprocess.run([soffice, "--headless", "--convert-to", "pdf", "--outdir", out_dir, docx_path],
                   check=True, capture_output=True, timeout=180)
    pdf_path = os.path.splitext(docx_path)[0] + ".pdf"
    return pdf_path if os.path.exists(pdf_path) else None

docx_filename = f"Weekly_Analytics_Brief_OLJ_{WEEK_START:%Y%m%d}_{WEEK_END:%Y%m%d}.docx"
build_brief_docx(docx_filename)
print(f"Saved {os.path.abspath(docx_filename)}")

pdf_filename = None
try:
    pdf_filename = export_pdf(docx_filename)
    print(f"Saved {os.path.abspath(pdf_filename)}" if pdf_filename else "[PDF skipped -- LibreOffice not installed]")
except Exception as e:
    print(f"[PDF export failed, .docx is fine] {type(e).__name__}: {e}")

EXPORTED_FILES = [f for f in (docx_filename, pdf_filename) if f]

if DOWNLOAD_IN_COLAB:
    try:
        from google.colab import files as colab_files
        for f in EXPORTED_FILES:
            colab_files.download(f)
    except ImportError:
        pass  # not in Colab -- files are saved next to the notebook

## 6. Send the email (only after you've checked the export)

Asks for a **y/N confirmation** before sending when run by hand (sends right away once you say yes).

**Who gets it:** test runs (Colab, or a manual *Run workflow*) go only to `TEST_RECIPIENTS`; the
scheduled Monday send goes to `SCHEDULED_RECIPIENTS`, or to the GitHub variable `BRIEF_RECIPIENTS` if set
(both lists are in ⚙️ Settings).

**Scheduled run:** `.github/workflows/weekly_brief.yml` runs this whole notebook every **Monday
morning** (Beirut time, summer and winter handled) with `SEND_EMAIL=1`. When the notebook finishes, this
cell **waits until 09:15** and sends, so the email always lands at 9:15 even if the run took a few
minutes. Change the time with `SEND_AT` below; the workflow starts about 35 minutes earlier.

**Gmail setup (one-time — App Password, not your regular password):**
1. The sending account needs **2-Step Verification** turned on (myaccount.google.com/security).
2. Go to **myaccount.google.com/apppasswords** → create one for "Mail" → copy the 16-character password.
3. Set `GMAIL_ADDRESS` and `GMAIL_APP_PASSWORD` as environment variables, or it'll prompt you.

⚠️ Untested against a real send from here. If Gmail rejects the login, double-check it's an
**App Password** (normal passwords are rejected once 2-Step Verification is on).

In [ ]:
import getpass
import time
from zoneinfo import ZoneInfo
import smtplib
from email import encoders
from email.mime.base import MIMEBase
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

IS_SCHEDULED = os.environ.get("GITHUB_EVENT_NAME") == "schedule"   # set by GitHub Actions itself

def _split_emails(text):
    return [e.strip() for e in str(text).replace(";", ",").replace("\n", ",").split(",") if e.strip()]

if IS_SCHEDULED:
    RECIPIENTS = _split_emails(os.environ.get("BRIEF_RECIPIENTS", "")) or SCHEDULED_RECIPIENTS
else:
    RECIPIENTS = TEST_RECIPIENTS
_bad = [e for e in RECIPIENTS if "@" not in e]
assert RECIPIENTS and not _bad, f"Check the recipient list -- empty or invalid: {_bad or RECIPIENTS}"
print(f"{'Scheduled send' if IS_SCHEDULED else 'Test send'} -> {', '.join(RECIPIENTS)}" + ("  (as BCC)" if EMAIL_AS_BCC else ""))

def send_email_with_attachments(recipients, subject, body, attachment_paths, bcc=False):
    sender = os.environ.get("GMAIL_ADDRESS") or input("Sending Gmail address: ")
    app_password = os.environ.get("GMAIL_APP_PASSWORD") or getpass.getpass("Gmail App Password (not your normal password): ")

    msg = MIMEMultipart()
    msg["From"], msg["Subject"] = sender, subject
    msg["To"] = sender if bcc else ", ".join(recipients)   # BCC: addresses only go in the envelope below
    msg.attach(MIMEText(body, "plain"))
    for path in attachment_paths:
        with open(path, "rb") as f:
            part = MIMEBase("application", "octet-stream")
            part.set_payload(f.read())
        encoders.encode_base64(part)
        part.add_header("Content-Disposition", f'attachment; filename="{os.path.basename(path)}"')
        msg.attach(part)

    with smtplib.SMTP("smtp.gmail.com", 587) as server:
        server.starttls()
        server.login(sender, app_password)
        server.send_message(msg, from_addr=sender, to_addrs=list(recipients))

SEND_AT = dt.time(9, 15)          # Beirut time. Scheduled (GitHub) runs wait until then before sending
SEND_TZ = ZoneInfo("Asia/Beirut")

def build_email(files):
    """Subject + body of the weekly mail -- edit the wording here."""
    fmt = "Word + PDF" if any(f.endswith(".pdf") for f in files) else "Word"
    subject = f"📊 OLJ Weekly Brief · {WEEK_START:%b %d}–{WEEK_END:%b %d}"
    body = f"""Hello from the Direction Numérique 👋

This week's OLJ Analytics Brief ({WEEK_START:%b %d} – {WEEK_END:%b %d, %Y}) is attached ({fmt}): one page, Web · App · Total.

🔎 One thing to watch: {watch_line}

🤖 This email is automated and sent every Monday at {SEND_AT:%H:%M}. Something looks off? Just let us know.

Direction Numérique · L'Orient-Le Jour
"""
    return subject, body

def wait_until_send_time():
    """Scheduled runs start a bit early (GitHub's scheduler can lag); hold the email until SEND_AT."""
    now = dt.datetime.now(SEND_TZ)
    target = now.replace(hour=SEND_AT.hour, minute=SEND_AT.minute, second=0, microsecond=0)
    wait = (target - now).total_seconds()
    if 0 < wait <= 3 * 3600:
        print(f"Waiting until {SEND_AT:%H:%M} Beirut time to send ({wait / 60:.0f} min)...")
        time.sleep(wait)

IN_CI = os.environ.get("GITHUB_ACTIONS") == "true"
SEND_EMAIL = os.environ.get("SEND_EMAIL", "").strip().lower() in ("1", "true", "yes")
if not SEND_EMAIL and not IN_CI:
    print("Exported:", ", ".join(EXPORTED_FILES))
    _subj, _body = build_email(EXPORTED_FILES)
    print(f"\n--- Email preview ---\nSubject: {_subj}\n\n{_body}")
    SEND_EMAIL = input(f"Checked it? Send to {', '.join(RECIPIENTS)} now? [y/N] ").strip().lower() == "y"

if SEND_EMAIL:
    if IN_CI and os.environ.get("SEND_NOW") != "1":   # manual "send now" runs skip the wait
        wait_until_send_time()
    try:
        email_subject, email_body = build_email(EXPORTED_FILES)
        send_email_with_attachments(RECIPIENTS, email_subject, email_body, EXPORTED_FILES, bcc=EMAIL_AS_BCC)
        print(f"Emailed to {len(RECIPIENTS)} recipient(s): {', '.join(RECIPIENTS)}")
    except Exception as e:
        print(f"Email step failed -- paste this error back and I'll adjust.\n{type(e).__name__}: {e}")
else:
    print("Not sent.")